In [ ]:
%load_ext autoreload
%autoreload 2

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "4"
 
print(os.getcwd())
project_root = os.getcwd()
while not os.path.exists(os.path.join(project_root, "pyproject.toml")) and project_root != os.path.dirname(project_root):
    project_root = os.path.dirname(project_root)
os.chdir(project_root)
print(os.getcwd())

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick 
import numpy as np
from model_utils.model_config import get_model_path
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ["CUDA_VISIBLE_DEVICES"] = "4"

In [ ]:
model_name_list = ["qwen3-4b", "qwen3-8b", "qwen3-14b", "toolace-2.5-8b", "watt-tool-8b"]

model_display_name_dict = {
    "qwen3-4b": "Qwen3-4B",
    "qwen3-8b": "Qwen3-8B",
    "qwen3-14b": "Qwen3-14B",
    "toolace-2.5-8b": "ToolACE-2.5-8B",
    "watt-tool-8b": "Watt-Tool-8B"
}

# Fig 6 uses D_1 with 10 chunks of 10 heads each (top-100 attention weights).
file_name = "pathway_add_1_trunc_500_test_chunk_10_n_10.json"


In [ ]:
result = {}
for model_name in model_name_list:
    file_path = os.path.join("results", model_name, "evaluate_compare", file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        result[model_display_name_dict[model_name]] = json.load(f)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re


def plot_paper_quality_chart(model_data, model_name):
    """
    Comparison of pathway strengths (Fig 6).
    
    """

    plt.rcParams['text.usetex'] = False 
    
    
    plt.rcParams['mathtext.fontset'] = 'cm'
    
    
    plt.rcParams['font.family'] = 'sans-serif'
    
    
    
    plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica']
    # sns.set_theme(style="ticks", font_scale=1.1, rc={"font.family": "serif"})
    
    # Data prep
    chunk_keys = list(model_data.keys())
    
    chunk_keys.sort(key=lambda x: int(re.search(r'start_(\d+)_', x).group(1)))
    
    
    x_indices = np.arange(len(chunk_keys))
    
    
    x_labels = []
    for k in chunk_keys:
        
        match = re.search(r'start_(\d+)_end_(\d+)', k)
        if match:
            start_val = int(match.group(1))
            end_val = int(match.group(2))
            
            x_labels.append(f"{start_val + 1}-{end_val}")
        else:
            
            x_labels.append(k)
    

    target = "avg_logit_diff_fix"
    
    
    sem_no = [model_data[k]['sem_patch_nocall_data'][target] for k in chunk_keys]
    sem_call = [model_data[k]['sem_patch_call_data'][target] for k in chunk_keys]
    str_no = [-model_data[k]['str_patch_nocall_data'][target] for k in chunk_keys]
    str_call = [-model_data[k]['str_patch_call_data'][target] for k in chunk_keys]

    # Plot setup
    
    sns.set_theme(style="ticks", font_scale=1.5, rc={"font.family": "sans-serif"})
    
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13,6), sharey=False)
    
    
    
    
    c_nocall = '#005f99' 
    c_call = '#d62728'   
    gg = "#333333"
    
    # ==========================
    # Left: Semantic pathway
    # ==========================
    
    ax1.plot(x_indices, sem_no, color=c_nocall, marker='o', markersize=6, 
             linewidth=2.8, label='No Invocation', zorder=3)
    ax1.plot(x_indices, sem_call, color=c_call, marker='^', markersize=6, 
             linewidth=2.8, linestyle='-', label='Invocation', zorder=3)
    
    
    ax1.fill_between(x_indices, sem_no, sem_call, color=gg, alpha=0.09)

    
    ax1.set_title(f'Semantic Pathways', fontsize=24, fontweight='bold', pad=13)


    ax1.set_ylabel(r'Semantic Checking Metric ($\Delta m_\text{sem}$)', fontweight='bold', fontsize=19) 
    ax1.set_xlabel('Top Attn Weights',fontsize=22, fontweight='bold')


    
    # ==========================
    # Right: Structural pathway
    # ==========================
    ax2.plot(x_indices, str_no, color=c_nocall, marker='o', markersize=6, 
             linewidth=2.8, label='No Invocation', zorder=3)
    ax2.plot(x_indices, str_call, color=c_call, marker='^', markersize=6, 
             linewidth=2.8, linestyle='-', label='Invocation', zorder=3)
    
    
    ax2.fill_between(x_indices, str_no, str_call, color=gg, alpha=0.09)
    
    ax2.set_title(f'Structural Pathways', fontsize=24, fontweight='bold', pad=13)
    ax2.set_xlabel('Top Attn Weights', fontsize=22, fontweight='bold')
    
    # ax2.set_ylabel('Logit Difference') 
    ax2.set_ylabel(r'Structural Matching Metric (${\Delta  m_\text{str}}$)', fontweight='bold', fontsize=19)

    # ==========================
    
    # ==========================
    for ax in [ax1, ax2]:
        
        ax.set_xticks(x_indices)
        ax.set_xticklabels(x_labels, rotation=30, fontsize=16)
        
        
        ax.grid(axis='y', linestyle='--', alpha=0.4, color='gray', zorder=0)
        
        
        # sns.despine(ax=ax)
        
        ax.legend(frameon=False, loc='best')

    
    # plt.suptitle(f'Model Analysis: {model_name}', fontsize=14, y=1.05, color='#333333')
    
    plt.tight_layout()
    
    
    safe_name = model_name.replace(" ", "_").replace(".", "-")
    plt.savefig(f'figs/mech/mech_compare_{safe_name}.pdf', bbox_inches='tight', dpi=300)
    plt.show()

In [ ]:
for model_name in model_name_list:
    display_name = model_display_name_dict[model_name]
    print(display_name)
    plot_paper_quality_chart(result[display_name], display_name)

In [ ]:
# Fig 18: per-D_k pathway ablation using top-2% attention heads (paper section 5.2).
file_name = "degree_data_{add_id}_trunc_500_test_chunk_{topn}.json"
topn_dict = {
    "qwen3-4b": 23,
    "qwen3-8b": 23,
    "qwen3-14b": 32,
    "toolace-2.5-8b": 20,
    "watt-tool-8b": 20,
}

for model_name in model_name_list:
    result[model_display_name_dict[model_name]] = {}
    for add_id in [0, 1, 2, 3, 4]:
        file_path = os.path.join("results", model_name, "evaluate_compare",
                                 file_name.format(add_id=add_id, topn=topn_dict[model_name]))
        with open(file_path, "r", encoding="utf-8") as f:
            result[model_display_name_dict[model_name]][add_id] = json.load(f)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.patheffects as path_effects

def plot_add_trend_dual_axis(model_result_dict, model_name, topn=10):
    """
    Dual-axis plot: semantic (left) vs. structural (right) strength over D_k.
    
    
    
    
    """
    
    # ==========================
    
    # ==========================
    plt.rcParams['text.usetex'] = False 
    plt.rcParams['mathtext.fontset'] = 'cm'
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica']
    
    sns.set_theme(style="white", font_scale=1.5, rc={"font.family": "sans-serif"})
    
    # ==========================
    
    # ==========================
    add_ids = [0, 1, 2, 3, 4]
    sem_values = []
    str_values = []
    
    target_layer = f"start_0_end_{topn}"
    target_metric = "avg_logit_diff_fix"
    
    
    if not model_result_dict:
        sem_values = [5, 4, 3, 2, 1] 
        str_values = [1, 2, 3, 4, 5]
    else:
        for aid in add_ids:
            if aid not in model_result_dict:
                sem_values.append(np.nan)
                str_values.append(np.nan)
                continue
            data = model_result_dict[aid][target_layer]
            sem_values.append(data['sem_patch_all_pair_data'][target_metric])
            str_values.append(-data['str_patch_all_pair_data'][target_metric])

    # ==========================
    
    # ==========================
    fig, ax1 = plt.subplots(figsize=(7, 6))
    
    
    c_sem = '#005f99'  
    c_str = '#d62728'  
    
    # Semantic (left axis)
    line1 = ax1.plot(add_ids, sem_values, color=c_sem, 
                     marker='o', markersize=10, 
                     linewidth=3, markeredgecolor='black', markeredgewidth=1.5,
                     label='Semantic (Left)')
    
    
    ax1.set_xlabel(r"Number of Added Attribute-Parameter", fontsize=19, fontweight='bold')
    ax1.set_ylabel(r"Semantic Checking Metric ($\Delta m_\text{sem}$)", color='black', fontsize=19, fontweight='bold')
    
    
    ax1.tick_params(axis='y', labelcolor='black', labelsize=14, width=2, color='black')
    ax1.tick_params(axis='x', labelsize=16, width=2, color='black')
    ax1.set_xticks(add_ids)

    ax1.set_title(f"Model: {model_name}", fontsize=21, fontweight='bold', pad=34)
    
    # Structural (right axis)
    ax2 = ax1.twinx() 
    
    line2 = ax2.plot(add_ids, str_values, color=c_str, 
                     marker='s', markersize=10, 
                     linewidth=3, markeredgecolor='black', markeredgewidth=1.5,
                     label='Structural (Right)')
    
    
    ax2.set_ylabel(r"Structural Matching Metric ($\Delta m_\text{str}$)", color='black', fontsize=19, fontweight='bold')
    
    ax2.tick_params(axis='y', labelcolor='black', labelsize=14, width=2, color='black')
    
    # ==========================
    
    # ==========================
    
    
    
    ax1.grid(axis='y', linestyle='--', alpha=0.3, color='gray')
    
    
    ax1.spines['left'].set_color('black'); ax1.spines['left'].set_linewidth(2)
    ax1.spines['bottom'].set_color('black'); ax1.spines['bottom'].set_linewidth(2)
    ax1.spines['top'].set_visible(False) 
    ax1.spines['right'].set_visible(False)
    
    
    ax2.spines['right'].set_color('black'); ax2.spines['right'].set_linewidth(2)
    ax2.spines['top'].set_color('black'); ax2.spines['top'].set_linewidth(2)
    ax2.spines['left'].set_visible(False)
    ax2.spines['bottom'].set_visible(False)

    
    # ax1.set_title(f"Component Decay Trend: {model_name}", fontsize=20, fontweight='bold', pad=35)

    # ==========================
    
    # ==========================
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    
    ax1.legend(lines, labels, 
               loc='lower center', 
               bbox_to_anchor=(0.5, 0.98), 
               ncol=2, 
               frameon=False, 
               fontsize=15)

    plt.tight_layout()
    
    safe_name = f"{model_name}_dual_axis".replace(" ", "_")
    plt.savefig(f'figs/mech/trend_dual/{safe_name}.pdf', bbox_inches='tight', dpi=300)
    plt.show()

In [ ]:

for model_name in model_name_list:

    display_name = model_display_name_dict[model_name]
    print(display_name)
    if display_name in result:
        plot_add_trend_dual_axis(result[display_name], display_name, topn=topn_dict[model_name])